# Project A5: Optimization Human Preference & LLM-as-a-Judge
This notebook demonstrates Direct Preference Optimization (DPO) and LLM-as-a-judge evaluation.

In [ ]:

%pip install -q -U transformers peft trl datasets huggingface_hub alpaca_eval accelerate bitsandbytes torchvision matplotlib seaborn pandas google-generativeai


## Environment & Runtime Check
Detects if the notebook is running in Google Colab, Local Mac (MPS), or Windows/Linux (CUDA).

In [ ]:

import torch
import sys
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"

def check_environment():
    in_colab = 'google.colab' in sys.modules
    
    if in_colab:
        print("Environment: Google Colab Detected.")
        # In Colab, we typically want to mount drive or just rely on the local session storage
        # from google.colab import drive
        # drive.mount('/content/drive')
    else:
        print("Environment: Local OS Detected.")
        
    if torch.cuda.is_available():
        return torch.device("cuda"), "CUDA is available. Operating on GPU."
    elif torch.backends.mps.is_available():
        return torch.device("mps"), "MPS is available. Operating on Apple Silicon GPU."
    else:
        return torch.device("cpu"), "No GPU available. Operating on CPU."

device, msg = check_environment()
print(msg)


## Task 1. Dataset Preparation
Load the dataset `jondurbin/truthy-dpo-v0.1` using the Hugging Face datasets library.

In [ ]:

from datasets import load_dataset

dataset_name = "jondurbin/truthy-dpo-v0.1"
print(f"Loading dataset: {dataset_name}")
dataset = load_dataset(dataset_name)

print(dataset)
print("\nSample Entry:")
print(dataset['train'][0])


## Task 2. Training a Model with DPOTrainer
Fine-tune a pre-trained transformer model using the train set.

In [ ]:

from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import DPOTrainer, DPOConfig

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Depending on the system memory, 4-bit quantization can be loaded. 
# For Apple Silicon (MPS), we may need to load in standard precision or bfloat16.
try:
    if device.type == "cuda":
        quantization_config = BitsAndBytesConfig(load_in_4bit=True)
        model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=quantization_config, device_map="auto")
        ref_model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=quantization_config, device_map="auto")
    else:
        model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.bfloat16).to(device)
        ref_model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.bfloat16).to(device)
        ref_model.eval()
except Exception as e:
    print(f"Error loading model: {e}\nFalling back to standard loading...")
    model = AutoModelForCausalLM.from_pretrained(model_name)
    ref_model = AutoModelForCausalLM.from_pretrained(model_name)

# LoRA Configuration
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)
# Note: DO NOT wrap model with get_peft_model here.
# TRL's DPOTrainer will automatically handle it when peft_config is passed.

training_args = DPOConfig(
    output_dir="./dpo_model_output",
    beta=0.1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=100,
    remove_unused_columns=False,
    optim="adamw_torch", # Safe optimizer block
    max_length=512
)

# For demonstration, subset the training data to save time.
train_subset = dataset['train'].select(range(100))

# Filter overlong prompts per TRL >0.28.0 recommendation
def filter_prompt_length(example):
    return len(tokenizer(example["prompt"])["input_ids"]) <= 256

train_subset = train_subset.filter(filter_prompt_length)

dpo_trainer = DPOTrainer(
    model,
    ref_model,
    args=training_args,
    train_dataset=train_subset,
    processing_class=tokenizer,
    peft_config=peft_config
)


In [ ]:

# Execute the training
print("Starting DPOTrainer....")
dpo_trainer.train()
dpo_trainer.save_model("./final_dpo_model")
print("Training Complete & Checkpoint Saved!")

# Plot Training Loss
import matplotlib.pyplot as plt
import seaborn as sns

log_history = dpo_trainer.state.log_history
# Extract loss and step
steps = [entry['step'] for entry in log_history if 'loss' in entry]
loss = [entry['loss'] for entry in log_history if 'loss' in entry]

if loss:
    plt.figure(figsize=(10, 5))
    sns.set_theme(style="whitegrid")
    sns.lineplot(x=steps, y=loss, marker="o", color="blue", linewidth=2)
    plt.title("DPO Training Loss Over Steps", fontsize=14)
    plt.xlabel("Training Steps", fontsize=12)
    plt.ylabel("Loss", fontsize=12)
    plt.show()


## Task 3. Pushing the Model to Hugging Face Hub
Using the local `huggingface-cli` environment variables from macOS.

In [ ]:

# Since you have huggingface-cli installed and authenticated on this Mac, 
# you can use HfApi to push the directory directly without needing notebook_login()!
from huggingface_hub import HfApi

# Create the API instance (pulls the token from your ~/.cache/huggingface/token)
api = HfApi()

model_repo_id = "my-nlp-a5-dpo-qwen" # Replace with your target repo ID (e.g. your_username/A5_Truthy_DPO)

print(f"Preparing to push to hub repository: {model_repo_id}")

try:
    # This pushes the trained adapter weights natively
    dpo_trainer.push_to_hub(model_repo_id)
    print("Push complete!")
except Exception as e:
    print("Could not push to Hub. Ensure the repo name includes your username, or that it is created first.")
    print(f"Error: {e}")


## Task 4. Evaluation: LLM-as-a-Judge
Load AlpacaEval, generate responses using Base and DPO model, and evaluate with an LLM-as-a-judge.

In [ ]:

import json
import random

# Load the AlpacaEval raw JSON
data_url = "https://huggingface.co/datasets/tatsu-lab/alpaca_eval/resolve/main/alpaca_eval.json"
alpaca_dataset = load_dataset("json", data_files=data_url)
helpful_base = alpaca_dataset['train'].filter(lambda x: x['dataset'] == 'helpful_base')

# Sample 15 items
sampled_data = list(helpful_base)
random.seed(42)
sampled_data = random.sample(sampled_data, 15)

print(f"Sampled {len(sampled_data)} prompts from helpful_base.")


In [ ]:

def generate_response(prompt_text, pipeline_model, tokenizer, device):
    inputs = tokenizer(prompt_text, return_tensors="pt").to(device)
    outputs = pipeline_model.generate(**inputs, max_new_tokens=100)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Generate responses (Placeholder loop)
# For real evaluation, ensure the models are properly loaded and generate outputs.
results = []
for item in sampled_data:
    instruction = item['instruction']
    
    # Placeholder: Replace with actual inference calls
    base_answer = "[BASE MODEL GENERATED TEXT]"
    dpo_answer = "[DPO MODEL GENERATED TEXT]"
    
    results.append({
        "instruction": instruction,
        "base_answer": base_answer,
        "dpo_answer": dpo_answer
    })


### LLM Judge Inference
Set up API calls for Gemini API to act as a judge.

In [ ]:

import google.generativeai as genai
import time

# Configure Gemini API
# NOTE: In production, store your key in environment variables or Colab Secrets.
import os

# Configure Gemini API securely across environments (Colab & Local Mac)
GEMINI_API_KEY = None

try:
    # Try Google Colab Secrets first
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except ImportError:
    # Not running in Colab
    pass
except userdata.SecretNotFoundError:
    # Running in colab but secret is missing
    pass

# Try Local OS Environment Variable
if not GEMINI_API_KEY:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

# As a final fallback, prompt the user dynamically
if not GEMINI_API_KEY:
    import getpass
    print("GEMINI_API_KEY not found in Colab Secrets or macOS Environment Variables.")
genai.configure(api_key=GEMINI_API_KEY)

# Use the latest Flash model for fast, cost-effective evaluation
gemini_model = genai.GenerativeModel("gemini-1.5-flash")

def llm_judge(instruction, base_answer, dpo_answer):
    prompt = f"""You are a highly qualified and impartial judge evaluating two AI models. Your task is to determine which model provides a better, more accurate, and more helpful response to the user's instruction.
User Instruction: {instruction}
Model A (Base Model): {base_answer}
Model B (DPO Model): {dpo_answer}
Evaluate both models. Output ONLY your final verdict as exactly one of the following options, with no extra text or explanation: "Model A", "Model B", or "Tie".
"""
    try:
        response = gemini_model.generate_content(prompt)
        verdict = response.text.strip().replace("`", "").replace('"', '')
        
        # Standardize Output
        if "Model A" in verdict:
            return "Model A"
        elif "Model B" in verdict:
            return "Model B"
        else:
            return "Tie"
    except Exception as e:
        print(f"Error calling Gemini: {e}")
        return "Tie"

evaluations = []
print("Evaluating with Gemini-1.5-Flash Judge...")
for i, res in enumerate(results):
    verdict = llm_judge(res['instruction'], res['base_answer'], res['dpo_answer'])
    evaluations.append({
        "instruction": res['instruction'],
        "verdict": verdict
    })
    print(f"[{i+1}/{len(results)}] Verdict: {verdict}")
    time.sleep(1) # Prevent rate limiting


In [ ]:

# Report
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

model_b_wins = sum(1 for e in evaluations if "Model B" in e['verdict'])
ties = sum(1 for e in evaluations if "Tie" in e['verdict'])
total = len(evaluations)

win_rate = (model_b_wins + (0.5 * ties)) / (total) * 100

print(f"Total Evaluations: {total}")
print(f"Model B (DPO) Wins: {model_b_wins}")
print(f"Ties: {ties}")
print(f"Win Rate: {win_rate:.2f}%")

df = pd.DataFrame(evaluations)
display(df)

# Visualize Win Rate
plt.figure(figsize=(8, 6))
labels = ['Model B (DPO) Wins', 'Ties', 'Model A (Base) Wins']
counts = [model_b_wins, ties, total - model_b_wins - ties]
colors = ['#38bdf8', '#cbd5e1', '#f87171']

plt.pie(counts, labels=labels, autopct='%1.1f%%', startangle=140, colors=colors, 
        wedgeprops={'edgecolor': 'white', 'linewidth': 2})
plt.title("LLM-as-a-Judge Evaluation Results", fontsize=16)
plt.axis('equal') # Equal aspect ratio ensures that pie is drawn as a circle.
plt.show()
